# AXIOM Constitutional AI — TinyLlama Fine-Tune

Fine-tunes TinyLlama-1.1B-Chat on 180 AXIOM constitutional AI examples using QLoRA.

**Pipeline:** Upload data → Load model (4-bit) → LoRA adapters → Train → Merge → GGUF → Download

**Requirements:** Colab T4 GPU (free tier), ~10 minutes training time

**Source:** github.com/Orivael-Dev/axiom | Patent Pending ORVL-001-PROV

In [ ]:
# Cell 1: Verify GPU
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU detected.\n"
        "Go to: Runtime > Change runtime type > T4 GPU"
    )

gpu_name = torch.cuda.get_device_name(0)
vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9 # Corrected line
print(f"GPU: {gpu_name}")
print(f"VRAM: {vram_gb:.1f} GB")
print(f"CUDA: {torch.version.cuda}")

GPU: Tesla T4
VRAM: 15.6 GB
CUDA: 12.8


In [ ]:
# Cell 2: Install dependencies
# bitsandbytes unpinned — Colab CUDA version changes frequently
!pip install -q transformers==4.46.0 peft==0.13.0 trl==0.12.0
!pip install -q bitsandbytes accelerate==1.0.0
!pip install -q datasets scipy sentencepiece protobuf

# Verify bitsandbytes can find CUDA
import bitsandbytes as bnb
import torch
print(f"bitsandbytes: {bnb.__version__}")
print(f"CUDA (torch): {torch.version.cuda}")
print(f"bnb CUDA setup: OK")
# Pull the AXIOM Colab adapter so Cell 3 can call load_training_data().
# Source: notebooks/axiom_colab.py + notebooks/sample_training_data.jsonl
!curl -fsSL -o axiom_colab.py https://raw.githubusercontent.com/Orivael-Dev/axiom/main/notebooks/axiom_colab.py
!curl -fsSL -o sample_training_data.jsonl https://raw.githubusercontent.com/Orivael-Dev/axiom/main/notebooks/sample_training_data.jsonl


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 3.5 MB/s eta 0:00:00
Reason for being yanked: This version unfortunately does not work with 3.8 but we did not drop the support yet
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 45.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.5/322.5 kB 32.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 310.2/310.2 kB 25.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 49.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 118.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 330.9/330.9 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.5 MB/s eta 0:00:00
bitsandbytes: 0.49.2
CUDA (torch): 12.8
bnb CUDA setup: OK


In [ ]:
# Cell 3: Load training data via the AXIOM Colab adapter.
#
# Same loader as the Qwen notebook, but `output_format='text'` for
# TinyLlama (Cell 4 reads ex['text']). Source auto-picked:
#   - Drive if mounted at /content/drive/MyDrive/axiom_training_data.jsonl
#   - Colab file picker otherwise
#   - bundled sample as last resort
#
# Explicit overrides:
#   load_training_data('upload', output_format='text')
#   load_training_data('drive:/some_folder/train.jsonl', output_format='text')
#   load_training_data('hf:tatsu-lab/alpaca', output_format='text')
#   load_training_data('sample', output_format='text')
from axiom_colab import load_training_data

examples = load_training_data(output_format='text')

print(f"\n{len(examples)} ChatML examples ready")
print(f"Preview:\n{examples[0]['text'][:300]}...")


In [ ]:
# Cell 4: Prepare HuggingFace Dataset — 90/10 train/eval split
from datasets import Dataset

dataset = Dataset.from_list(examples)
dataset = dataset.shuffle(seed=42)

split = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = split["train"]
eval_dataset  = split["test"]

print(f"Train: {len(train_dataset)} examples")
print(f"Eval:  {len(eval_dataset)} examples")

# Token length check
lengths = [len(ex["text"]) for ex in examples]
print(f"\nChar lengths — min: {min(lengths)}, max: {max(lengths)}, median: {sorted(lengths)[len(lengths)//2]}")
print(f"Est tokens — total: {sum(lengths)//4:,}, avg: {sum(lengths)//len(lengths)//4}")

Train: 261 examples
Eval:  29 examples

Char lengths — min: 586, max: 4399, median: 1477
Est tokens — total: 142,372, avg: 490


In [ ]:
# Cell 5: Load TinyLlama in 4-bit (QLoRA)
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print(f"Model: {MODEL_NAME}")
print(f"Parameters: {model.num_parameters():,}")
print(f"VRAM used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

Model: TinyLlama/TinyLlama-1.1B-Chat-v1.0
Parameters: 1,100,048,384
VRAM used: 0.77 GB


In [ ]:
# Cell 6: Configure LoRA adapters
# rank=16, alpha=32 — enough capacity for domain shift, won't overfit 180 examples
# Targets: all 4 attention projections, skip MLP (unnecessary at this scale)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 4,505,600 || all params: 1,104,553,984 || trainable%: 0.4079


In [ ]:
# Cell 7: Training config
# 4 epochs, effective batch 8, cosine LR 2e-4, eval every 20 steps
# Est. ~5-10 minutes on T4
from trl import SFTTrainer, SFTConfig

sft_config = SFTConfig(
    output_dir="./axiom-tinyllama-lora",

    # Duration
    num_train_epochs=4,

    # Batch (effective = 1 * 8 = 8)
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    per_device_eval_batch_size=1,

    # Learning rate
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,

    # Sequence
    max_seq_length=1024,

    # Precision (T4 supports fp16, not bf16)
    fp16=True,
    bf16=False,

    # Evaluation
    eval_strategy="steps",
    eval_steps=20,

    # Checkpoints
    save_strategy="steps",
    save_steps=20,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    # Logging
    logging_steps=5,
    report_to="none",

    # Optimizer
    optim="paged_adamw_8bit",
    weight_decay=0.05,
    max_grad_norm=0.3,

    # Dataset
    dataset_text_field="text",
    packing=False,
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    tokenizer=tokenizer,
)

print(f"Training steps: {trainer.state.max_steps if hasattr(trainer.state, 'max_steps') else 'TBD'}")
print(f"Ready to train.")

Map:   0%|          | 0/261 [00:00<?, ? examples/s]

Map:   0%|          | 0/29 [00:00<?, ? examples/s]

Training steps: 0
Ready to train.


In [ ]:
# Cell 8: Train
import time

t0 = time.time()
trainer.train()
elapsed = time.time() - t0

print(f"\nTraining complete in {elapsed/60:.1f} minutes")
print(f"Final train loss: {trainer.state.log_history[-1].get('train_loss', 'N/A')}")
print(f"Best eval loss:   {min(h.get('eval_loss', 999) for h in trainer.state.log_history if 'eval_loss' in h):.4f}")

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss,Validation Loss
20,2.039900,1.650326
40,1.385600,1.171516
60,1.240100,1.069840
80,1.265900,1.032541
100,1.281600,1.012692
120,1.054600,1.008301


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/pyt


Training complete in 9.5 minutes
Final train loss: 1.434489380568266
Best eval loss:   1.0083


In [ ]:
# Cell 9: Save LoRA adapters
trainer.save_model("./axiom-tinyllama-lora/final")
tokenizer.save_pretrained("./axiom-tinyllama-lora/final")
print("LoRA adapters saved to ./axiom-tinyllama-lora/final")

import os
adapter_size = sum(
    os.path.getsize(os.path.join(dp, f))
    for dp, _, fns in os.walk("./axiom-tinyllama-lora/final")
    for f in fns
) / (1024 * 1024)
print(f"Adapter size: {adapter_size:.1f} MB")

LoRA adapters saved to ./axiom-tinyllama-lora/final
Adapter size: 21.2 MB


In [ ]:
# Cell 10: Merge LoRA into base model
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# Reload base in float16 for clean merge
base_model = AutoModelForCausalLM.from_pretrained(
    "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    torch_dtype=torch.float16,
    device_map="auto",
)

# Merge LoRA weights into base
model = PeftModel.from_pretrained(base_model, "./axiom-tinyllama-lora/final")
model = model.merge_and_unload()

# Save merged model
model.save_pretrained("./axiom-tinyllama-merged")
tok = AutoTokenizer.from_pretrained("./axiom-tinyllama-lora/final")
tok.save_pretrained("./axiom-tinyllama-merged")

merged_size = sum(
    os.path.getsize(os.path.join(dp, f))
    for dp, _, fns in os.walk("./axiom-tinyllama-merged")
    for f in fns
) / (1024**3)
print(f"Merged model saved: {merged_size:.2f} GB")

Merged model saved: 2.05 GB


In [ ]:
# Cell 11: Convert to GGUF Q4_K_M for Ollama deployment
import subprocess, os, shutil

# Clean up any partial clone from previous attempts
if os.path.exists("llama.cpp") and not os.path.exists("llama.cpp/convert_hf_to_gguf.py"):
    print("Removing incomplete llama.cpp clone...")
    shutil.rmtree("llama.cpp")

# Install GGUF conversion tools
!pip install -q gguf sentencepiece

# Clone llama.cpp (retry-safe)
if not os.path.exists("llama.cpp"):
    print("Cloning llama.cpp...")
    result = subprocess.run(
        ["git", "clone", "--depth", "1", "https://github.com/ggerganov/llama.cpp.git"],
        capture_output=True, text=True, timeout=120
    )
    if result.returncode != 0:
        raise RuntimeError(f"git clone failed: {result.stderr}")

assert os.path.exists("llama.cpp/convert_hf_to_gguf.py"), \
    "convert_hf_to_gguf.py not found — clone may have failed"
print("llama.cpp ready")

# Step 1: HF → GGUF F16
print("\n[1/3] Converting to GGUF F16...")
!python llama.cpp/convert_hf_to_gguf.py \
    ./axiom-tinyllama-merged \
    --outfile axiom-tinyllama-f16.gguf \
    --outtype f16

assert os.path.exists("axiom-tinyllama-f16.gguf"), \
    "F16 conversion failed — check output above"
f16_mb = os.path.getsize("axiom-tinyllama-f16.gguf") / (1024 * 1024)
print(f"F16 GGUF: {f16_mb:.0f} MB")

# Step 2: Build quantize tool
print("\n[2/3] Building llama-quantize...")
!cd llama.cpp && cmake -B build -DCMAKE_BUILD_TYPE=Release 2>&1 | tail -3 && \
    cmake --build build --target llama-quantize -j$(nproc) 2>&1 | tail -5

QUANTIZE_BIN = "llama.cpp/build/bin/llama-quantize"
assert os.path.exists(QUANTIZE_BIN), \
    "llama-quantize build failed — check cmake output above"
print("llama-quantize built")

# Step 3: Quantize to Q4_K_M (~600-700 MB)
print("\n[3/3] Quantizing to Q4_K_M...")
!./llama.cpp/build/bin/llama-quantize \
    axiom-tinyllama-f16.gguf \
    axiom-tinyllama-Q4_K_M.gguf \
    Q4_K_M

assert os.path.exists("axiom-tinyllama-Q4_K_M.gguf"), \
    "Quantization failed — check output above"
size_mb = os.path.getsize("axiom-tinyllama-Q4_K_M.gguf") / (1024 * 1024)
print(f"\nFinal: axiom-tinyllama-Q4_K_M.gguf ({size_mb:.0f} MB)")
print("Ready for Cell 12 download")

In [ ]:
# Cell 12: Download GGUF — save to models/ in your repo
from google.colab import files

print("Downloading axiom-tinyllama-Q4_K_M.gguf...")
print("Save to: i:/vsCode/promt-agent/models/")
files.download("axiom-tinyllama-Q4_K_M.gguf")

print("\nDeployment:")
print("  cd models/")
print("  ollama create axiom-tinyllama -f Modelfile")
print("  ollama run axiom-tinyllama")

Save to: i:/vsCode/promt-agent/models/


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Deployment:
  cd models/
  ollama create axiom-tinyllama -f Modelfile
  ollama run axiom-tinyllama
